# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I went with a **line chart**, because the question is about change over time and a line lets you follow each continent's path from 1970 to 2022 at a glance. Since the continents start at very different sizes (Asia has billions, Oceania only tens of millions), I indexed every continent to **1970 = 100**, so we compare growth rather than raw size. Africa gets the one accent colour (orange) since it is the answer to the question, while the other five continents stay grey as context. I labelled the lines directly at the end and dropped the legend, so nobody has to look back and forth.

In [3]:
# Q1
# Total population per continent and year, then index to 1970 = 100
cont = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
               .sort_values(['Continent', 'Year']))
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first') * 100

ACCENT, GREY = '#D55E00', '#B3B3B3'          # colour-blind-safe orange + neutral grey
colors = {c: (ACCENT if c == 'Africa' else GREY) for c in cont['Continent'].unique()}

fig = px.line(cont, x='Year', y='Index', color='Continent',
              color_discrete_map=colors, template='plotly_white',
              title='Africa has quadrupled since 1970 — no other continent has even come close')

fig.update_traces(line_width=2)
fig.update_traces(line_width=4, selector=dict(name='Africa'))

# Direct labels at the end of each line instead of a legend
last = cont[cont['Year'] == 2022]
for _, r in last.iterrows():
    fig.add_annotation(x=2022, y=r['Index'], text=f"{r['Continent']} ({r['Index']:.0f})",
                       xanchor='left', xshift=8, showarrow=False,
                       font=dict(color=colors[r['Continent']] if r['Continent'] == 'Africa' else '#666',
                                 size=12 if r['Continent'] != 'Africa' else 14))

fig.add_hline(y=100, line_dash='dot', line_color='#999')
fig.update_layout(showlegend=False, margin=dict(r=150),
                  yaxis_title='Population index (1970 = 100)', xaxis_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Here I used a **sorted horizontal bar chart**. We are ranking countries by one number (people added), and bars are the easiest thing for the eye to compare because they all start from the same zero line. Sorting from biggest to smallest makes the ranking readable immediately, and horizontal bars give the country names room to breathe. Africa is highlighted in orange and everything else is grey, so the "how many are in Africa?" part of the question can be answered by just counting orange bars. I removed the gridlines and put the values at the end of the bars instead.

In [4]:
# Q2
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').sort_values('Added')          # ascending so the biggest ends up on top
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
n_africa = (top15['Continent'] == 'Africa').sum()

fig = px.bar(top15, x='Added', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa': '#D55E00', 'Other continents': '#B3B3B3'},
             text=top15['Added'] / 1e6, template='plotly_white',
             title=f'Only {n_africa} of the 15 biggest population gainers since 1970 are in Africa — '
                   'India and China lead by far')

fig.update_traces(texttemplate='%{text:.0f}M', textposition='outside', cliponaxis=False)
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title=None,
                  margin=dict(r=60))
fig.update_xaxes(visible=False, showgrid=False)
fig.update_yaxes(showgrid=False)
fig.add_annotation(x=0.98, y=0.12, xref='paper', yref='paper', showarrow=False, xanchor='right',
                   text='<b style="color:#D55E00">Orange</b> = Africa<br>Grey = other continents',
                   font=dict(size=13, color='#666'))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

This one is about the relationship between two numeric variables, so a **scatter plot** is the natural fit (each dot is a country). Both area and population span many orders of magnitude, from tiny islands to China and Russia, so I used **log scales** on both axes. Without that, almost all the dots would be squashed into one corner. Colour is used for the continent, with a colour-blind-safe palette, so we can see whether the pattern differs between them. I added a thin grey trend line to show the overall pattern (a clear but loose positive relationship), and I only labelled the countries that sit furthest from that line, so the chart is not buried in text.

In [5]:
# Q3
cb = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#009E73',   # Okabe-Ito colour-blind-safe palette
      'North America': '#CC79A7', 'Oceania': '#F0E442', 'South America': '#56B4E9'}

x, y = np.log10(df['Area']), np.log10(df['2022 Population'])
slope, intercept = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0, 1]
df['resid'] = y - (slope * x + intercept)

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 color_discrete_map=cb, hover_name='Country', log_x=True, log_y=True,
                 template='plotly_white',
                 title=f'Bigger countries tend to have more people (r = {r:.2f} on log scales), '
                       'but Russia, Canada and Australia are far emptier than their size suggests')

fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='white')))

# Overall trend line (grey)
xs = np.linspace(x.min(), x.max(), 50)
fig.add_scatter(x=10 ** xs, y=10 ** (slope * xs + intercept), mode='lines',
                line=dict(color='#888', dash='dash', width=1.5), showlegend=False, hoverinfo='skip')

# Label only the biggest outliers (furthest from the trend)
outliers = pd.concat([df.nlargest(3, 'resid'), df.nsmallest(5, 'resid')])
for _, o in outliers.iterrows():
    fig.add_annotation(x=np.log10(o['Area']), y=np.log10(o['2022 Population']), text=o['Country'],
                       showarrow=True, arrowhead=0, arrowcolor='#999', ax=25, ay=-20,
                       font=dict(size=11, color='#444'))

fig.update_layout(xaxis_title='Land area (km², log scale)', yaxis_title='Population 2022 (log scale)',
                  legend_title_text=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Since the question explicitly asks for the *spread* and not just the average, I chose a **box plot**, one box per continent. It shows the median, the middle half of the countries and the outliers in a compact way, so continents can be compared side by side. Density is extremely skewed (a few city-states are thousands of times denser than most countries), so I used a **log scale**, otherwise every box would be flattened at the bottom. The boxes are ordered by median density so the ranking is visible, and all are one calm blue because the continent is already on the x-axis, so extra colours would just be decoration. I labelled the most extreme countries directly.


In [6]:
# Q4
order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()

fig = px.box(df, x='Continent', y='Density', points='outliers', log_y=True,
             category_orders={'Continent': order}, hover_name='Country',
             template='plotly_white',
             title='Density varies enormously inside every continent — '
                   'a few city-states dwarf the typical country')

fig.update_traces(marker=dict(color='#0072B2', size=5, opacity=0.7),
                  line=dict(color='#0072B2'), fillcolor='rgba(0,114,178,0.25)')

# Label the most extreme countries
for _, o in df.nlargest(4, 'Density').iterrows():
    fig.add_annotation(x=o['Continent'], y=np.log10(o['Density']), text=o['Country'],
                       showarrow=False, xanchor='left', xshift=10, font=dict(size=11, color='#444'))

fig.update_layout(xaxis_title=None, yaxis_title='People per km² (2022, log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

he question asks *where in the world* countries grew or shrank, so a **choropleth map** is the right tool, because the geography is the point. I plotted the percentage change from 2000 to 2022 using a **diverging colour scale** with white at zero: blue for growth and orange for shrinkage. That avoids red/green, so it is safe for colour-blind readers, and the zero midpoint makes growth vs. decline obvious at a glance. I capped the colour range at +160% so a handful of extreme growers (like Gulf states) don't wash out everyone else. I also removed the map frame and kept the layout minimal.

In [7]:
# Q5
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
n_shrank = (df['Change'] < 0).sum()
lo, hi = -40, 160                                       # colour range (values are clipped to it)
df['Change_clipped'] = df['Change'].clip(lo, hi)

# Diverging scale with white exactly at 0%: orange = shrank, blue = grew
zero = (0 - lo) / (hi - lo)
scale = [[0, '#D55E00'], [zero, '#F7F7F7'], [1, '#0072B2']]

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change_clipped',
                    color_continuous_scale=scale, range_color=(lo, hi),
                    hover_name='Country', hover_data={'Change': ':.1f', 'Change_clipped': False, 'CCA3': False},
                    template='plotly_white',
                    title=f'Most of the world grew since 2000 — only {n_shrank} of {len(df)} countries shrank, '
                          'mostly in Europe and small island territories')

fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(coloraxis_colorbar=dict(title='Change 2000–22', ticksuffix='%',
                                          tickvals=[-40, 0, 50, 100, 160], ticktext=['-40%', '0%', '+50%', '+100%', '≥160%'],
                                          len=0.6, thickness=12),
                  margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

The question asks for a part-to-whole picture with *two levels* (continents, then the countries inside them) in a single figure, which is exactly what a **treemap** does. Each rectangle's area is proportional to population, so the giants (China, India) jump out and the tiny countries fade into the background, and the nesting shows which continent each country belongs to. I coloured by continent using the same colour-blind-safe palette as in Q3, so colour carries meaning here too. The labels show each country's share of the world total, and tiny rectangles are left unlabelled automatically so the figure stays readable.

In [8]:
# Q6
cb = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#009E73',
      'North America': '#CC79A7', 'Oceania': '#F0E442', 'South America': '#56B4E9'}

asia_share = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / df['2022 Population'].sum()

fig = px.treemap(df, path=['Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map=cb, template='plotly_white',
                 title=f'Asia is home to {asia_share:.0%} of the world — and China and India alone hold over a third')

fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}',
                  textfont=dict(size=13), root_color='white',
                  marker=dict(line=dict(width=1, color='white')))
fig.update_layout(margin=dict(t=60, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Because the question is about *shares of a whole* changing over time, I used a **100% stacked bar chart**, one bar per year. Every bar adds up to 100%, so the eye can focus purely on how the slices change. Africa's slice grows and Europe's shrinks, while Asia stays roughly flat at 58–61%. I grouped North America, South America and Oceania as 'Rest of the world' (in grey, as context), as the question suggested, and I used a colour-blind-safe palette for the three continents we care about. The percentages are printed on the bars and the series are labelled on the right, so no legend is needed.

In [9]:
# Q7
df_long['Group'] = np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                            df_long['Continent'], 'Rest of the world')

share = df_long.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
share['Share'] = share['Population'] / share.groupby('Year')['Population'].transform('sum') * 100

stack = ['Asia', 'Africa', 'Europe', 'Rest of the world']           # bottom -> top
cmap = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#56B4E9', 'Rest of the world': '#D9D9D9'}

af70 = share.query("Year == 1970 and Group == 'Africa'")['Share'].iloc[0]
af22 = share.query("Year == 2022 and Group == 'Africa'")['Share'].iloc[0]
eu70 = share.query("Year == 1970 and Group == 'Europe'")['Share'].iloc[0]
eu22 = share.query("Year == 2022 and Group == 'Europe'")['Share'].iloc[0]

fig = px.bar(share, x='Year', y='Share', color='Group', color_discrete_map=cmap,
             category_orders={'Group': stack}, template='plotly_white',
             title=f"Africa's share of humanity rose from {af70:.0f}% to {af22:.0f}% "
                   f"while Europe's fell from {eu70:.0f}% to {eu22:.0f}%")

fig.update_traces(texttemplate='%{y:.0f}%', textposition='inside', insidetextanchor='middle')
fig.update_xaxes(type='category', title=None, showgrid=False)       # years are unevenly spaced -> treat as labels
fig.update_yaxes(visible=False, showgrid=False)

# Direct labels on the right instead of a legend
last = share[share['Year'] == 2022].set_index('Group')['Share']
bottom = 0
for g in stack:
    fig.add_annotation(x=1.01, xref='paper', y=bottom + last[g] / 2, yref='y', text=g,
                       showarrow=False, xanchor='left', font=dict(size=13, color='#444'))
    bottom += last[g]

fig.update_layout(showlegend=False, margin=dict(r=140), bargap=0.15)
fig.show()